In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, recall_score
import warnings

warnings.filterwarnings("ignore")

In [2]:
USE_URL       = "https://tfhub.dev/google/universal-sentence-encoder/4" # DAN version
BATCH_SIZE    = 32
EPOCHS        = 10
LEARNING_RATE = 1e-4
LSTM_UNITS    = 100
TEST_SIZE     = 0.2
RANDOM_SEED   = 42

In [3]:
file_path = r"C:\Users\chouh\OneDrive\Desktop\VirginiaTech\Spring 2026\SocialMediaAnalytics\FinalProject\archive\twitter_sentiment_data.csv"
df = pd.read_csv(file_path)

# Drop missing and filter for only Stance labels: -1 (Denier) and 1 (Believer)
df = df.dropna(subset=["message"])
stance_df = df[df["sentiment"].isin([-1, 1])].copy()
stance_df["stance_label"] = stance_df["sentiment"].map({-1: 0, 1: 1})

print(f"Climate Data Filtered: {len(stance_df):,} samples.")

Climate Data Filtered: 26,952 samples.


In [4]:
use_model = hub.load(USE_URL)

def encode_texts(texts, batch_size=256):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        emb = use_model(texts[i: i + batch_size]).numpy()
        embeddings.append(emb)
    return np.vstack(embeddings)

In [5]:
isarcasm_path = r"C:\Users\chouh\OneDrive\Desktop\VirginiaTech\Spring 2026\SocialMediaAnalytics\FinalProject\archive\train.En.csv"
isarcasm_df = pd.read_csv(isarcasm_path)

X_sarcasm = encode_texts(isarcasm_df["tweet"].astype(str).tolist())
X_sarcasm_seq = np.expand_dims(X_sarcasm, axis=1) # (N, 1, 512)
y_sarcasm = isarcasm_df["sarcastic"].values

In [6]:
s_inp = tf.keras.Input(shape=(1, 512))
s_bilstm_layer = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(LSTM_UNITS, return_sequences=False), name="sarcasm_bilstm"
)(s_inp)
s_out = tf.keras.layers.Dense(1, activation='sigmoid')(s_bilstm_layer)

s_expert = tf.keras.Model(s_inp, s_out)
s_expert.compile(optimizer='adam', loss='binary_crossentropy')
s_expert.fit(X_sarcasm_seq, y_sarcasm, epochs=5, batch_size=32, verbose=1)

# Extract Weights to "Teach" the Stance Model sarcasm
sarcasm_weights = s_expert.get_layer("sarcasm_bilstm").get_weights()
print("Sarcasm Knowledge Captured.")

Epoch 1/5
109/109 [==============================] - 2s 3ms/step - loss: 0.5844
Epoch 2/5
109/109 [==============================] - 0s 3ms/step - loss: 0.5411
Epoch 3/5
109/109 [==============================] - 0s 3ms/step - loss: 0.5230
Epoch 4/5
109/109 [==============================] - 0s 3ms/step - loss: 0.5110
Epoch 5/5
109/109 [==============================] - 0s 3ms/step - loss: 0.5028
Sarcasm Knowledge Captured.


In [7]:
X_stance = encode_texts(stance_df["message"].tolist())
X_stance_seq = np.expand_dims(X_stance, axis=1) # (N, 1, 512)
y_stance = stance_df["stance_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_stance_seq, y_stance, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_stance
)

In [9]:
def build_stance_model():
    inp = tf.keras.Input(shape=(1, 512), name="use_input")

    # CNN Layer: Extracts local feature signals from the USE vector
    x = tf.keras.layers.Conv1D(filters=128, kernel_size=1, activation='relu')(inp)
    x = tf.keras.layers.Dropout(0.2)(x)

    # BiLSTM Layer: With Sarcasm Weight Initialization
    bilstm_layer = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(LSTM_UNITS, return_sequences=False),
        name="bilstm_stance"
    )

    # Apply BiLSTM
    x = bilstm_layer(x)

    # Dense Classifier
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid', name="stance_output")(x)

    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_stance_model()

In [10]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/10
607/607 [==============================] - 4s 4ms/step - loss: 0.4270 - accuracy: 0.8501 - val_loss: 0.3475 - val_accuracy: 0.8530
Epoch 2/10
607/607 [==============================] - 2s 3ms/step - loss: 0.3294 - accuracy: 0.8595 - val_loss: 0.3102 - val_accuracy: 0.8674
Epoch 3/10
607/607 [==============================] - 2s 3ms/step - loss: 0.3045 - accuracy: 0.8745 - val_loss: 0.2961 - val_accuracy: 0.8771
Epoch 4/10
607/607 [==============================] - 2s 3ms/step - loss: 0.2918 - accuracy: 0.8819 - val_loss: 0.2881 - val_accuracy: 0.8827
Epoch 5/10
607/607 [==============================] - 2s 3ms/step - loss: 0.2837 - accuracy: 0.8851 - val_loss: 0.2825 - val_accuracy: 0.8873
Epoch 6/10
607/607 [==============================] - 2s 3ms/step - loss: 0.2753 - accuracy: 0.8886 - val_loss: 0.2777 - val_accuracy: 0.8850
Epoch 7/10
607/607 [==============================] - 2s 3ms/step - loss: 0.2671 - accuracy: 0.8937 - val_loss: 0.2710 - val_accuracy: 0.8906
Epoch 

In [11]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

acc = accuracy_score(y_test, y_pred) * 100
f1_macro = f1_score(y_test, y_pred, average="macro") * 100
rec_macro = recall_score(y_test, y_pred, average="macro") * 100

print("\n" + "="*50)
print(f"Stance Accuracy     : {acc:.2f}%")
print(f"Stance F1 (Macro)   : {f1_macro:.2f}%")
print(f"Stance Recall(Macro): {rec_macro:.2f}%")
print("="*50)

print("\nDetailed Stance Report:")
print(classification_report(y_test, y_pred, target_names=["Denier", "Believer"]))

169/169 [==============================] - 0s 847us/step

Stance Accuracy     : 89.06%
Stance F1 (Macro)   : 74.96%
Stance Recall(Macro): 71.83%

Detailed Stance Report:
              precision    recall  f1-score   support

      Denier       0.69      0.47      0.56       798
    Believer       0.91      0.96      0.94      4593

    accuracy                           0.89      5391
   macro avg       0.80      0.72      0.75      5391
weighted avg       0.88      0.89      0.88      5391

